# Assignment: A* Search on a Grid

## Objective
In this assignment you will implement and analyze the **A\* Search algorithm** on a grid-based pathfinding problem.

The goal is to understand how **heuristics guide search** and how they affect the number of explored nodes.

---

## Problem Description

You are given a **10 × 10 grid world** representing a map. Each cell in the grid can contain:

| Symbol | Meaning |
|------|------|
| `S` | Start position |
| `G` | Goal position |
| `.` | Free cell (traversable) |
| `#` | Obstacle (blocked cell) |

The agent can move in **four directions only**:

- Up
- Down
- Left
- Right

Each movement has a **uniform cost of 1**.

Your task is to compute the **shortest path from the Start (S) to the Goal (G)**.

---

## Algorithms to Compare

You will compare two algorithms:

### 1. Dijkstra’s Algorithm
Dijkstra’s algorithm explores nodes based purely on the **current path cost**:

\[
f(n) = g(n)
\]

Where:

- \(g(n)\) = cost from start to node \(n\)

It guarantees the shortest path but may explore many unnecessary nodes.

---

### 2. A\* Search Algorithm
A\* improves efficiency by adding a **heuristic estimate** of the remaining distance:

\[
f(n) = g(n) + h(n)
\]

Where:

- \(g(n)\) = cost from start to node \(n\)
- \(h(n)\) = heuristic estimate from node \(n\) to the goal

In this assignment we use the **Manhattan Distance heuristic**:

\[
h(n) = |x_n - x_g| + |y_n - y_g|
\]

This heuristic is **admissible** and **consistent** for grid movement without diagonals.

---

## Expected Output
An expected output for the given grid is shown in the Cell Output below the skeleton code.

In [1]:
import heapq


def build_path(goal, parent):
    path = []
    curr = goal
    while curr is not None:
        path.append(curr)
        curr = parent[curr[0]][curr[1]]

    path.reverse()
    return path


def assist_dijkstra(pq, dist, parent, curr_node, next_node):  # push_in_queue
    cX, cY = curr_node
    nX, nY = next_node
    new_distance = dist[cX][cY] + 1
    next_node_distance = dist[nX][nY]

    if next_node_distance > new_distance:
        parent[nX][nY] = (cX, cY)
        dist[nX][nY] = new_distance
        heapq.heappush(pq, (new_distance, next_node))


def dijkstra(grid, src, goal):
    pq = []
    cost = 0
    explored = 0
    isGoalFound = False
    directions = [(0, 1), (0, -1), (-1, 0), (1, 0)]

    rows = len(grid)
    dist = [[float("inf") for _ in row] for row in grid]
    parent = [[None for _ in row] for row in grid]

    dist[src[0]][src[1]] = 0
    heapq.heappush(pq, (0, src))
    while pq:
        curr_weight, curr_node = heapq.heappop(pq)
        row, col = curr_node

        if curr_weight > dist[row][col]:
            continue
        explored = explored + 1
        if curr_node == goal:
            cost = dist[row][col]
            isGoalFound = True
            break
        for dr, dc in directions:
            next_row = row + dr
            next_col = col + dc

            if (
                0 <= next_row < rows
                and 0 <= next_col < len(grid[next_row])
                and grid[next_row][next_col] != "#"
            ):
                next_node = (next_row, next_col)
                assist_dijkstra(pq, dist, parent, curr_node, next_node)

    if isGoalFound:
        return build_path(goal, parent), cost, explored
    else:
        return None, None, explored


def get_heuristic(curr, goal):
    cX, cY = curr
    gx, gy = goal
    return abs(cX - gx) + abs(cY - gy)


def assist_a_star(pq, dist, parent, goal, curr_node, next_node):
    cX, cY = curr_node
    nX, nY = next_node
    new_distance = dist[cX][cY] + 1
    next_node_distance = dist[nX][nY]

    if next_node_distance > new_distance:
        parent[nX][nY] = (cX, cY)
        dist[nX][nY] = new_distance
        next_heu = get_heuristic(next_node, goal) + new_distance
        heapq.heappush(pq, (next_heu, new_distance, next_node))


def a_star(grid, src, goal):
    pq = []
    cost = 0
    explored = 0
    isGoalFound = False
    directions = [(0, 1), (0, -1), (-1, 0), (1, 0)]

    rows = len(grid)
    dist = [[float("inf") for _ in row] for row in grid]
    parent = [[None for _ in row] for row in grid]

    dist[src[0]][src[1]] = 0
    src_heu = get_heuristic(src, goal)
    heapq.heappush(pq, (src_heu, 0, src))
    while pq:
        _, curr_weight, curr_node = heapq.heappop(pq)
        row, col = curr_node

        if curr_weight > dist[row][col]:
            continue
        explored = explored + 1
        if curr_node == goal:
            cost = dist[row][col]
            isGoalFound = True
            break
        for dr, dc in directions:
            next_row = row + dr
            next_col = col + dc

            if (
                0 <= next_row < rows
                and 0 <= next_col < len(grid[next_row])
                and grid[next_row][next_col] != "#"
            ):
                next_node = (next_row, next_col)
                assist_a_star(pq, dist, parent, goal, curr_node, next_node)

    if isGoalFound:
        return build_path(goal, parent), cost, explored
    else:
        return None, None, explored


if __name__ == "__main__":
    grid = [
        ["S", ".", ".", ".", "#", ".", ".", ".", ".", "."],
        [".", "#", "#", ".", "#", ".", "#", "#", "#", "."],
        [".", ".", ".", ".", ".", ".", ".", ".", "#", "."],
        ["#", "#", ".", "#", "#", "#", ".", "#", "#", "."],
        [".", ".", ".", ".", ".", "#", ".", ".", ".", "."],
        [".", "#", "#", "#", ".", "#", "#", "#", ".", "#"],
        [".", ".", ".", "#", ".", ".", ".", "#", ".", "."],
        ["#", "#", ".", "#", "#", "#", ".", "#", "#", "."],
        [".", ".", ".", ".", ".", ".", ".", ".", "#", "."],
        [".", "#", "#", "#", "#", "#", ".", ".", ".", "G"],
    ]
    src = (0, 0)
    goal = (9, 9)
    path, cost, explored = dijkstra(grid, src, goal)
    print("DIJKSTRA RESULT")
    print("Path:", path)
    print("Cost:", cost)
    print("Nodes explored:", explored)
    print()

    path, cost, explored = a_star(grid, src, goal)
    print("A* RESULT")
    print("Path:", path)
    print("Cost:", cost)
    print("Nodes explored:", explored)

DIJKSTRA RESULT
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (4, 7), (4, 8), (5, 8), (6, 8), (6, 9), (7, 9), (8, 9), (9, 9)]
Cost: 18
Nodes explored: 62

A* RESULT
Path: [(0, 0), (0, 1), (0, 2), (0, 3), (1, 3), (2, 3), (2, 4), (2, 5), (2, 6), (3, 6), (4, 6), (4, 7), (4, 8), (5, 8), (6, 8), (6, 9), (7, 9), (8, 9), (9, 9)]
Cost: 18
Nodes explored: 39
